In [ ]:
# ============================================================
# BLOCK 3: MODEL INTERPRETATION
# Project: Forecasting Household Deposit Volume in Russia
# Author: Nadezhda Silkina
# Date: 2026
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import cross_val_score
import shap
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries loaded")

# ============================================================
# 2. LOAD DATA AND PREPARATION
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

# Create features (lags)
df['DEPOS_log'] = np.log(df['DEPOS'])
for lag in [1, 3, 6, 12]:
    df[f'DEPOS_lag_{lag}'] = df['DEPOS'].shift(lag)

# Prepare X and y
X = df.drop(['DEPOS', 'DEPOS_log'], axis=1).dropna()
y = df.loc[X.index, 'DEPOS']

# Split into train/test
train_size = len(X) - 12
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

# Scaling (for Ridge and Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data loaded. Records: {len(df)}")
print(f"📊 X_train shape: {X_train.shape}")
print(f"📊 X_test shape: {X_test.shape}")

# ============================================================
# 3. SHAP ANALYSIS (FOR BEST MODEL — RIDGE)
# ============================================================

print("\n" + "="*60)
print("3. SHAP ANALYSIS (RIDGE REGRESSION)")
print("="*60)

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)

print("\n✅ Ridge model trained")

# SHAP analysis
print("\n🔍 Calculating SHAP values...")
explainer = shap.LinearExplainer(ridge_model, X_train_scaled, feature_names=X.columns)
shap_values = explainer.shap_values(X_test_scaled)

# Top-5 features by SHAP
shap_importance = pd.DataFrame({
    'feature': X.columns,
    'shap_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print("\n📊 TOP-5 FEATURES BY SHAP (RIDGE):")
print(shap_importance.head(5).to_string(index=False))

# 3.3. SHAP visualization
print("\n📊 SHAP visualization...")

# Plot 1: Global feature importance
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns, plot_type="bar", show=False)
plt.title('Global Feature Importance (SHAP)', fontsize=14)
plt.tight_layout()
plt.savefig('03_shap_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 2: Direction of influence
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns, show=False)
plt.title('Direction of Feature Influence (SHAP)', fontsize=14)
plt.tight_layout()
plt.savefig('03_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 3: SHAP dependence on feature value (for top-1)
top_feature = shap_importance.iloc[0]['feature']
plt.figure(figsize=(8, 6))
shap.dependence_plot(top_feature, shap_values, X_test_scaled, feature_names=X.columns, show=False)
plt.title(f'SHAP Dependence on {top_feature}', fontsize=14)
plt.tight_layout()
plt.savefig('03_shap_dependence.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ SHAP analysis completed")

# ============================================================
# 4. NONLINEAR RELATIONSHIP ANALYSIS
# ============================================================

print("\n" + "="*60)
print("4. NONLINEAR RELATIONSHIP ANALYSIS")
print("="*60)

def calculate_full_metrics(y_train, y_test, y_train_pred, y_test_pred, n_params):
    """
    Calculate comprehensive metrics on both training and test sets.

    Parameters:
    -----------
    y_train, y_test : array-like
        Actual values
    y_train_pred, y_test_pred : array-like
        Predicted values
    n_params : int
        Number of model parameters (including intercept)

    Returns:
    --------
    dict with metrics including overfitting gap
    """
    n_train = len(y_train)

    # Training set metrics
    r2_train = r2_score(y_train, y_train_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))

    # Adjusted R² on training set
    if n_train - n_params - 1 > 0:
        r2_adj_train = 1 - (1 - r2_train) * (n_train - 1) / (n_train - n_params - 1)
    else:
        r2_adj_train = np.nan

    # AIC and BIC on training set (unbiased variance estimator)
    residuals_train = y_train - y_train_pred
    rss_train = np.sum(residuals_train**2)
    sigma2 = rss_train / (n_train - n_params)
    log_likelihood = -0.5 * n_train * (np.log(2 * np.pi * sigma2) + 1)

    aic = -2 * log_likelihood + 2 * n_params
    bic = -2 * log_likelihood + n_params * np.log(n_train)

    # Test set metrics
    r2_test = r2_score(y_test, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

    # Overfitting gap: R²_train - R²_test
    r2_gap = r2_train - r2_test

    return {
        'n_params': n_params,
        'r2_train': r2_train,
        'r2_adj_train': r2_adj_train,
        'aic': aic,
        'bic': bic,
        'r2_test': r2_test,
        'rmse_test': rmse_test,
        'r2_gap': r2_gap  # Difference R²_train - R²_test (larger means stronger overfitting)
    }

# ============================================================
# 4.1. Baseline model (Linear Regression)
# ============================================================

print("\n🔍 Baseline linear model (for comparison):")
lr_base = LinearRegression()
lr_base.fit(X_train, y_train)

y_train_pred_base = lr_base.predict(X_train)
y_test_pred_base = lr_base.predict(X_test)

n_params_base = X_train.shape[1] + 1
metrics_base = calculate_full_metrics(
    y_train, y_test,
    y_train_pred_base, y_test_pred_base,
    n_params_base
)

print(f"   R²_train = {metrics_base['r2_train']:.4f}, R²_test = {metrics_base['r2_test']:.4f}")
print(f"   (full metrics — in comparison table below)")

# ============================================================
# 4.2. Ramsey test (RESET test)
# ============================================================

print("\n🔍 Ramsey test (checking for omitted nonlinearities)...")
print("   Method: add y_pred² and y_pred³ to the model and check their significance")

# Add powers of predicted values
X_train_reset = X_train.copy()
X_train_reset['y_pred2'] = y_train_pred_base ** 2
X_train_reset['y_pred3'] = y_train_pred_base ** 3

# Train model with added powers
lr_reset = LinearRegression()
lr_reset.fit(X_train_reset, y_train)

# Check significance of added features
from sklearn.feature_selection import f_regression
f_values, p_values = f_regression(X_train_reset[['y_pred2', 'y_pred3']], y_train)

print(f"\n📊 Ramsey test results:")
print(f"   y_pred²: F = {f_values[0]:.4f}, p = {p_values[0]:.4f}")
print(f"   y_pred³: F = {f_values[1]:.4f}, p = {p_values[1]:.4f}")

if p_values[0] < 0.05 or p_values[1] < 0.05:
    print("   📌 CONCLUSION: Omitted nonlinearities detected (p < 0.05)")
    print("   → Checking whether polynomial features improve model quality")
else:
    print("   📌 CONCLUSION: No omitted nonlinearities detected (p >= 0.05)")

# ============================================================
# 4.3. Training alternative models
# ============================================================

print("\n" + "-"*60)
print("TRAINING ALTERNATIVE MODELS")
print("-"*60)

# --- Model 1: All polynomials (degree=2) ---
print("\n🔧 Model 1: All polynomials (degree=2)")
print("   Description: All quadratic features and interactions (104 features)")

poly_all = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_train_poly_all = poly_all.fit_transform(X_train)
X_test_poly_all = poly_all.transform(X_test)

lr_poly_all = LinearRegression()
lr_poly_all.fit(X_train_poly_all, y_train)

y_train_pred_poly = lr_poly_all.predict(X_train_poly_all)
y_test_pred_poly = lr_poly_all.predict(X_test_poly_all)

metrics_poly = calculate_full_metrics(
    y_train, y_test,
    y_train_pred_poly, y_test_pred_poly,
    X_train_poly_all.shape[1] + 1
)

# --- Model 2: Lasso regularization ---
print("\n🔧 Model 2: Lasso regularization")
print("   Description: L1-regularization for automatic feature selection from polynomials")

lasso = Lasso(alpha=0.01, max_iter=10000)
lasso.fit(X_train_poly_all, y_train)

n_nonzero = np.sum(np.abs(lasso.coef_) > 1e-6)
print(f"   Features selected: {n_nonzero} out of {X_train_poly_all.shape[1]}")

y_train_pred_lasso = lasso.predict(X_train_poly_all)
y_test_pred_lasso = lasso.predict(X_test_poly_all)

metrics_lasso = calculate_full_metrics(
    y_train, y_test,
    y_train_pred_lasso, y_test_pred_lasso,
    n_nonzero + 1  # only non-zero coefficients + intercept
)

# --- Model 3: EDA hypotheses ---
print("\n🔧 Model 3: Features from EDA hypotheses")
print("   Description: DEP1², UNEM², UNEM×DEP1, SERV² (17 features)")

X_train_hyp = X_train.copy()
X_test_hyp = X_test.copy()

X_train_hyp['DEP1_sq'] = X_train_hyp['DEP1'] ** 2
X_test_hyp['DEP1_sq'] = X_test_hyp['DEP1'] ** 2

X_train_hyp['UNEM_sq'] = X_train_hyp['UNEM'] ** 2
X_test_hyp['UNEM_sq'] = X_test_hyp['UNEM'] ** 2

X_train_hyp['UNEM_DEP1'] = X_train_hyp['UNEM'] * X_train_hyp['DEP1']
X_test_hyp['UNEM_DEP1'] = X_test_hyp['UNEM'] * X_test_hyp['DEP1']

X_train_hyp['SERV_sq'] = X_train_hyp['SERV'] ** 2
X_test_hyp['SERV_sq'] = X_test_hyp['SERV'] ** 2

lr_hyp = LinearRegression()
lr_hyp.fit(X_train_hyp, y_train)

y_train_pred_hyp = lr_hyp.predict(X_train_hyp)
y_test_pred_hyp = lr_hyp.predict(X_test_hyp)

metrics_hyp = calculate_full_metrics(
    y_train, y_test,
    y_train_pred_hyp, y_test_pred_hyp,
    X_train_hyp.shape[1] + 1
)

# --- Model 4: Stepwise Selection ---
print("\n🔧 Model 4: Stepwise Selection")
print("   Description: Sequential selection of 10 best features with cross-validation")

sfs = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select=10,
    direction='forward',
    scoring='r2',
    cv=5,
    n_jobs=-1
)

sfs.fit(X_train_hyp, y_train)
selected_features = X_train_hyp.columns[sfs.get_support()].tolist()

print(f"   Features selected: {len(selected_features)}")
print(f"   Features: {selected_features}")

X_train_sfs = X_train_hyp[selected_features]
X_test_sfs = X_test_hyp[selected_features]

lr_sfs = LinearRegression()
lr_sfs.fit(X_train_sfs, y_train)

y_train_pred_sfs = lr_sfs.predict(X_train_sfs)
y_test_pred_sfs = lr_sfs.predict(X_test_sfs)

metrics_sfs = calculate_full_metrics(
    y_train, y_test,
    y_train_pred_sfs, y_test_pred_sfs,
    X_train_sfs.shape[1] + 1
)

# ============================================================
# 4.4. COMPARISON TABLE OF ALL MODELS
# ============================================================

print("\n" + "="*60)
print("📊 COMPARISON TABLE OF ALL MODELS")
print("="*60)

models_comparison = pd.DataFrame({
    'Model': [
        'Baseline (Linear)',
        'All polynomials',
        'Lasso (selection)',
        'EDA hypotheses',
        'Stepwise (selection)'
    ],
    'Features': [
        metrics_base['n_params'] - 1,
        metrics_poly['n_params'] - 1,
        metrics_lasso['n_params'] - 1,
        metrics_hyp['n_params'] - 1,
        metrics_sfs['n_params'] - 1
    ],
    'R²_train': [
        metrics_base['r2_train'],
        metrics_poly['r2_train'],
        metrics_lasso['r2_train'],
        metrics_hyp['r2_train'],
        metrics_sfs['r2_train']
    ],
    'R²_adj_train': [
        metrics_base['r2_adj_train'],
        metrics_poly['r2_adj_train'],
        metrics_lasso['r2_adj_train'],
        metrics_hyp['r2_adj_train'],
        metrics_sfs['r2_adj_train']
    ],
    'AIC': [
        metrics_base['aic'],
        metrics_poly['aic'],
        metrics_lasso['aic'],
        metrics_hyp['aic'],
        metrics_sfs['aic']
    ],
    'BIC': [
        metrics_base['bic'],
        metrics_poly['bic'],
        metrics_lasso['bic'],
        metrics_hyp['bic'],
        metrics_sfs['bic']
    ],
    'R²_test': [
        metrics_base['r2_test'],
        metrics_poly['r2_test'],
        metrics_lasso['r2_test'],
        metrics_hyp['r2_test'],
        metrics_sfs['r2_test']
    ],
    'RMSE_test': [
        metrics_base['rmse_test'],
        metrics_poly['rmse_test'],
        metrics_lasso['rmse_test'],
        metrics_hyp['rmse_test'],
        metrics_sfs['rmse_test']
    ],
    'R²_gap': [
        metrics_base['r2_gap'],
        metrics_poly['r2_gap'],
        metrics_lasso['r2_gap'],
        metrics_hyp['r2_gap'],
        metrics_sfs['r2_gap']
    ]
})

print("\n📊 Metrics on TRAINING sample (n=122):")
print(models_comparison[['Model', 'Features', 'R²_train', 'R²_adj_train', 'AIC', 'BIC']].round(4).to_string(index=False))

print("\n📊 Metrics on TEST sample (n=12):")
print(models_comparison[['Model', 'R²_test', 'RMSE_test', 'R²_gap']].round(4).to_string(index=False))

print("\n📌 Note: R²_gap = R²_train - R²_test — difference between quality")
print("   on training and test samples. Large R²_gap indicates")
print("   model overfitting.")

# ============================================================
# 4.5. TESTING EACH HYPOTHESIS SEPARATELY
# ============================================================

print("\n" + "-"*60)
print("DETAILED TESTING OF EDA HYPOTHESES")
print("-"*60)
print("Each hypothesis is tested separately: add one feature")
print("to the baseline model and observe metric changes.")

def test_hypothesis_full(feature_name, y_train, y_test, metrics_base):
    """
    Test a single hypothesis by adding one feature to base model.
    """
    X_train_test = X_train.copy()
    X_test_test = X_test.copy()
    X_train_test[feature_name] = X_train_hyp[feature_name]
    X_test_test[feature_name] = X_test_hyp[feature_name]

    lr_test = LinearRegression()
    lr_test.fit(X_train_test, y_train)

    y_train_pred_test = lr_test.predict(X_train_test)
    y_test_pred_test = lr_test.predict(X_test_test)

    n_params_test = X_train_test.shape[1] + 1
    metrics_test = calculate_full_metrics(
        y_train, y_test,
        y_train_pred_test, y_test_pred_test,
        n_params_test
    )

    return {
        'feature': feature_name,
        'r2_train_improvement': metrics_test['r2_train'] - metrics_base['r2_train'],
        'r2_test_improvement': metrics_test['r2_test'] - metrics_base['r2_test'],
        'aic_change': metrics_test['aic'] - metrics_base['aic'],
        'r2_gap_change': metrics_test['r2_gap'] - metrics_base['r2_gap']
    }

hypotheses = {
    'DEP1²': 'DEP1_sq',
    'UNEM²': 'UNEM_sq',
    'UNEM × DEP1': 'UNEM_DEP1',
    'SERV²': 'SERV_sq'
}

hypothesis_results = []
for name, feature in hypotheses.items():
    result = test_hypothesis_full(feature, y_train, y_test, metrics_base)
    result['hypothesis_name'] = name
    hypothesis_results.append(result)

# Create DataFrame for clarity
hyp_df = pd.DataFrame(hypothesis_results)
hyp_df = hyp_df[['hypothesis_name', 'feature', 'r2_train_improvement', 'r2_test_improvement', 'aic_change', 'r2_gap_change']]
hyp_df.columns = ['Hypothesis', 'Feature', 'ΔR²_train', 'ΔR²_test', 'ΔAIC', 'ΔR²_gap']

print("\n📊 Hypothesis testing results:")
print(hyp_df.round(4).to_string(index=False))

print("\n📌 Interpretation:")
print("   ΔR²_train > 0 — model better explains training data")
print("   ΔR²_test > 0 — model better predicts new data")
print("   ΔAIC < 0 — model is better considering complexity (penalty for parameters)")
print("   ΔR²_gap > 0 — overfitting increased")

# Text explanations for each hypothesis
print("\n📝 Conclusions for each hypothesis:")
for _, row in hyp_df.iterrows():
    name = row['Hypothesis']
    dr2_train = row['ΔR²_train']
    dr2_test = row['ΔR²_test']
    daic = row['ΔAIC']
    dgap = row['ΔR²_gap']

    print(f"\n   {name}:")
    if dr2_test > 0.001 and daic < 0:
        print(f"      ✅ CONFIRMED: improves both prediction (ΔR²_test = {dr2_test:+.4f}),")
        print(f"         and information criterion (ΔAIC = {daic:+.2f})")
    elif dr2_test > 0.001:
        print(f"      ⚠️ PARTIALLY: improves prediction (ΔR²_test = {dr2_test:+.4f}),")
        print(f"         but AIC worsened (ΔAIC = {daic:+.2f}) — quality improvement")
        print(f"         does not compensate for model complexity")
    elif dr2_train > 0.001 and dr2_test <= 0.001:
        print(f"      ❌ NOT CONFIRMED: improves only training sample")
        print(f"         (ΔR²_train = {dr2_train:+.4f}), but not test sample")
        print(f"         (ΔR²_test = {dr2_test:+.4f}) — feature does not generalize")
    else:
        print(f"      ❌ NOT CONFIRMED: no improvement on either sample")
        print(f"         (ΔR²_train = {dr2_train:+.4f}, ΔR²_test = {dr2_test:+.4f})")

# ============================================================
# 4.6. FINAL SUMMARY
# ============================================================

print("\n" + "="*60)
print("📌 FINAL SUMMARY ON NONLINEARITY ANALYSIS")
print("="*60)

# Select best model by different criteria
best_by_r2_train = models_comparison.loc[models_comparison['R²_train'].idxmax()]
best_by_r2_test = models_comparison.loc[models_comparison['R²_test'].idxmax()]
best_by_aic = models_comparison.loc[models_comparison['AIC'].idxmin()]
best_by_bic = models_comparison.loc[models_comparison['BIC'].idxmin()]

print(f"\n🏆 Best models by different criteria:")
print(f"   By R²_train (goodness of fit): {best_by_r2_train['Model']} ({best_by_r2_train['R²_train']:.4f})")
print(f"   By R²_test (predictive ability): {best_by_r2_test['Model']} ({best_by_r2_test['R²_test']:.4f})")
print(f"   By AIC (balance quality/complexity): {best_by_aic['Model']} ({best_by_aic['AIC']:.2f})")
print(f"   By BIC (with stronger penalty): {best_by_bic['Model']} ({best_by_bic['BIC']:.2f})")

print(f"\n📊 Overfitting analysis (R²_gap = R²_train - R²_test):")
for _, row in models_comparison.iterrows():
    gap = row['R²_gap']
    if gap > 5.0:
        status = "❌ CRITICAL"
    elif gap > 1.0:
        status = "❌ STRONG"
    elif gap > 0.2:
        status = "⚠️ Moderate"
    elif gap > 0.1:
        status = "ℹ️ Minor"
    else:
        status = "✅ Good generalization"
    print(f"   {row['Model']}: gap = {gap:.4f} → {status}")

print(f"\n📌 KEY FINDINGS:")
print(f"   1. Ramsey test detected nonlinearities (p < 0.05)")
print(f"   2. But adding polynomials leads to CRITICAL overfitting:")
print(f"      - All polynomials: R²_train = {metrics_poly['r2_train']:.4f}, R²_test = {metrics_poly['r2_test']:.4f}")
print(f"      - Lasso: R²_train = {metrics_lasso['r2_train']:.4f}, R²_test = {metrics_lasso['r2_test']:.4f}")
print(f"   3. EDA hypotheses do not improve prediction:")
print(f"      - UNEM² and UNEM×DEP1 give slight R²_test improvement but worsen AIC")
print(f"      - DEP1² and SERV² give no improvement")
print(f"   4. Baseline linear model remains optimal with R²_test = {metrics_base['r2_test']:.4f}")
print(f"   5. Nonlinearities exist, but their direct addition to the model")
print(f"      leads to overfitting due to small sample size (n=122)")

print("\n✅ Block 3 completed")